# Figure 4 — the posterior is not a fit, it is what survives

Prior, then 2, 4 and 6 observations. Sample paths thin out, the credible band collapses, the mean sharpens. The mean posterior sd is printed on each panel so the collapse is a number, not just a picture.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

xs = np.linspace(0, 10, 500)
OX = np.array([1.15, 2.90, 4.30, 6.10, 7.40, 8.90])
OY = land.f1d(OX)

fig, axes = plt.subplots(4, 1, figsize=(style.FIG_W_FULL, 5.6), sharex=True,
                         gridspec_kw=dict(hspace=0.22))
labels = ["prior — before any experiment", "n = 2", "n = 4", "n = 6"]

for ax, k, lab in zip(axes, [0, 2, 4, 6], labels):
    g = gpmod.GP(gpmod.matern52, ls=0.85, sf=1.0, sn=0.03)
    if k:
        g.fit(OX[:k, None], OY[:k])
    mu, sd = g.predict(xs[:, None])
    for s in g.sample(xs[:, None], n=60, seed=100 + k):
        ax.plot(xs, s, color=style.TEAL, lw=0.35, alpha=0.22, zorder=2)
    ax.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=style.TEAL,
                    alpha=0.13, lw=0, zorder=3)
    ax.plot(xs, mu, color=style.INK, lw=1.6, zorder=5,
            alpha=0.55 if k == 0 else 1.0)
    if k:
        ax.plot(OX[:k], OY[:k], "o", ms=6, color=style.RED,
                mec="white", mew=1.0, zorder=7)
    ax.set_ylim(-2.6, 2.6)
    ax.set_yticks([-2, 0, 2])
    ax.text(0.008, 0.90, lab, transform=ax.transAxes, va="top",
            fontsize=10, fontweight="bold",
            color=style.RED if k else style.INK)
    ax.text(0.992, 0.90, f"mean sd = {sd.mean():.3f}", transform=ax.transAxes,
            va="top", ha="right", fontsize=9.5, color=style.GRAY)

axes[-1].set_xlabel("reaction parameter  x")
axes[0].set_title("Each observation deletes functions from the ensemble — "
                  "nothing is fitted", loc="left", color=style.INK)
style.save(fig, "fig_04_posterior_collapse", OUT)